In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [2]:
df = pd.read_csv('for_model_selection_train_df.csv')

In [3]:
df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,pooja_room,study_room,servant_room,store_room,others,furnishing_type,luxury_category,floor_category,NearbyLocationsCategory
0,0,flat,sohna road,0.35,2,2,2,Relatively New,556.0,0,0,0,0,0,0,Low,Mid Floor,Low
1,1,flat,sector 56,2.05,4,4,3+,Moderately Old,2667.0,0,0,0,0,0,0,Low,Mid Floor,High
2,2,flat,sector 90,1.25,3,3,3+,Relatively New,1700.0,1,0,1,0,0,0,Medium,High Floor,Low
3,3,flat,sector 56,1.45,3,3,3+,Old Property,1900.0,1,1,1,0,1,0,High,Mid Floor,High
4,4,flat,sector 92,0.66,3,2,2,Relatively New,1473.0,0,0,0,0,0,1,Low,High Floor,Low


In [4]:
df['furnishing_type'].value_counts()

furnishing_type
0    3218
1     336
Name: count, dtype: int64

In [6]:
# 0 -> unfurnished
# 1 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0:'unfurnished',1:'furnished'})

In [7]:
df.head()

,Unnamed: 0,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,pooja_room,study_room,servant_room,store_room,others,furnishing_type,luxury_category,floor_category,NearbyLocationsCategory
0,0,flat,sohna road,0.35,2,2,2,Relatively New,556.0,0,0,0,0,0,unfurnished,Low,Mid Floor,Low
1,1,flat,sector 56,2.05,4,4,3+,Moderately Old,2667.0,0,0,0,0,0,unfurnished,Low,Mid Floor,High
2,2,flat,sector 90,1.25,3,3,3+,Relatively New,1700.0,1,0,1,0,0,unfurnished,Medium,High Floor,Low
3,3,flat,sector 56,1.45,3,3,3+,Old Property,1900.0,1,1,1,0,1,unfurnished,High,Mid Floor,High
4,4,flat,sector 92,0.66,3,2,2,Relatively New,1473.0,0,0,0,0,0,furnished,Low,High Floor,Low


In [14]:
df=df.drop(columns=['luxury_category','pooja_room','others','Unnamed: 0'])
X = df.drop(columns=['price'])
y = df['price']

In [15]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

## Ordinal Encoding 

In [16]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category','NearbyLocationsCategory']

In [23]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room','store_room','study_room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
    unknown_value=-1), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [24]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [25]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [26]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [27]:
model_output

[['linear_reg', np.float64(0.7271309576371562), 0.803218660741711],
 ['svr', np.float64(0.7537985566901407), 0.7599182876824475],
 ['ridge', np.float64(0.7271371916137345), 0.8031541045580737],
 ['LASSO', np.float64(0.057510822276252614), 1.4058226220651493],
 ['decision tree', np.float64(0.8035999993536118), 0.6198560706221369],
 ['random forest', np.float64(0.8887480382308489), 0.4151939854137645],
 ['extra trees', np.float64(0.8745114622851498), 0.439939096538266],
 ['gradient boosting', np.float64(0.8773891939383873), 0.46660839699549456],
 ['adaboost', np.float64(0.7554621375787272), 0.7386405466621166],
 ['mlp', np.float64(0.7981430397930439), 0.692211864916722],
 ['xgboost', np.float64(0.8981105626172681), 0.40427076483191987]]

In [28]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [29]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.898111,0.404271
5,random forest,0.888748,0.415194
6,extra trees,0.874511,0.439939
7,gradient boosting,0.877389,0.466608
4,decision tree,0.803600,0.619856
9,mlp,0.798143,0.692212
8,adaboost,0.755462,0.738641
1,svr,0.753799,0.759918
2,ridge,0.727137,0.803154
0,linear_reg,0.727131,0.803219


## OHE

In [34]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category','NearbyLocationsCategory']
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
    unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',handle_unknown='ignore'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [35]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [36]:
model2_output = []
for model_name,model in model_dict.items():
    model2_output.append(scorer(model_name, model))

C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as 

In [37]:
model2_df = pd.DataFrame(model2_output, columns=['name','r2','mae'])

In [38]:
model2_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.895141,0.404397
6,extra trees,0.898993,0.405940
10,xgboost,0.900507,0.416700
7,gradient boosting,0.878040,0.474307
9,mlp,0.878880,0.482922
4,decision tree,0.821330,0.557747
2,ridge,0.851793,0.577373
0,linear_reg,0.851691,0.578123
1,svr,0.756702,0.756796
8,adaboost,0.756164,0.768259


## OHE With PCA

In [ ]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room','study_room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output


model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

model_df.sort_values(['mae'])

## Target Encoder

In [40]:
# we do it for cols which have large no of categories
# may be chance of dataleakage , so wehavae to do this after train test split
# if we do this before train test split then dataleakage is happens 

In [41]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category','NearbyLocationsCategory']


# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room','study_room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
        unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [42]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [43]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [44]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [45]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.907655,0.372479
10,xgboost,0.909941,0.380590
6,extra trees,0.904675,0.400782
7,gradient boosting,0.890430,0.446554
9,mlp,0.849427,0.558921
4,decision tree,0.831332,0.575064
8,adaboost,0.822801,0.628536
2,ridge,0.821947,0.665797
0,linear_reg,0.821927,0.665805
1,svr,0.770708,0.743807


## Hyperparameter Tuning

In [46]:
from sklearn.model_selection import GridSearchCV

In [58]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['auto', 'sqrt'] #'max_features': [1.0, 'sqrt', 'log2']
}

In [48]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category','NearbyLocationsCategory']


# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room','study_room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
        unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [51]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [52]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

In [53]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits


C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
640 fits failed out of a total of 1280.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
369 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\THARUN TEJA S\.conda\envs\aiml_env\Lib\si

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['auto', 'sqrt'], 'regressor__max_samples': [0.1, 0.25, ...], 'regressor__n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",KFold(n_split... shuffle=True)
,"verbose verbose: intControls the verbosity: the hi

In [54]:
final_pipe = search.best_estimator_

In [55]:
search.best_params_

{'regressor__max_depth': 20,
 'regressor__max_features': 'sqrt',
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [56]:
search.best_score_

np.float64(0.9055459952851802)

In [57]:
final_pipe.fit(X,y_transformed)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [59]:
## using xgboost with bayesian 

In [60]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error

from category_encoders import TargetEncoder

from skopt import BayesSearchCV
from skopt.space import Real, Integer

In [61]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category','NearbyLocationsCategory']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room','study_room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
        unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [62]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    random_state=42,
    n_jobs=1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb)
])

In [63]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_transformed,
    test_size=0.2,
    random_state=42
)

kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [64]:
search_spaces = {
    'regressor__n_estimators': Integer(100, 600),

    'regressor__max_depth': Integer(2, 10),

    'regressor__learning_rate': Real(
        0.01, 0.3, prior='log-uniform'
    ),

    'regressor__subsample': Real(0.6, 1.0),

    'regressor__colsample_bytree': Real(0.6, 1.0),

    'regressor__min_child_weight': Integer(1, 10),

    'regressor__gamma': Real(0, 5),

    'regressor__reg_alpha': Real(
        1e-8, 10, prior='log-uniform'
    ),

    'regressor__reg_lambda': Real(
        1e-3, 20, prior='log-uniform'
    )
}

In [65]:
bayes_search = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_spaces,
    n_iter=50,
    cv=kfold,
    scoring='r2',
    n_jobs=-1,
    n_points=1,
    random_state=42,
    verbose=2
)


In [66]:
bayes_search.fit(X_train, y_train)

Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for eac

,estimator,"Pipeline(step...=None, ...))])"
,search_spaces,"{'regressor__colsample_bytree': Real(low=0.6,...m='normalize'), 'regressor__gamma': Real(low=0, h...m='normalize'), 'regressor__learning_rate': Real(low=0.01...m='normalize'), 'regressor__max_depth': Integer(low=2...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,50
,scoring,'r2'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,KFold(n_split... shuffle=True)


In [67]:
print("Best Parameters:")
print(bayes_search.best_params_)

print("Best CV R2:")
print(bayes_search.best_score_)

Best Parameters:
OrderedDict([('regressor__colsample_bytree', 0.6), ('regressor__gamma', 0.0), ('regressor__learning_rate', 0.08451266031458327), ('regressor__max_depth', 6), ('regressor__min_child_weight', 1), ('regressor__n_estimators', 385), ('regressor__reg_alpha', 1e-08), ('regressor__reg_lambda', 13.71332547409398), ('regressor__subsample', 0.9321465953997092)])
Best CV R2:
0.9050925943390877


In [68]:
final_pipe = bayes_search.best_estimator_

In [69]:
y_pred = final_pipe.predict(X_test)

In [70]:
y_pred = np.expm1(y_pred)
y_test_original = np.expm1(y_test)

In [71]:
print("Test R2:", r2_score(y_test_original, y_pred))

print("Test MAE:", mean_absolute_error(
    y_test_original,
    y_pred
))

Test R2: 0.9077827457697389
Test MAE: 0.38846891447763404


In [72]:
# for RF
from sklearn.base import clone
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

final_rf = clone(pipeline_rf)

final_rf.set_params(**search.best_params_)

final_rf.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [74]:
rf_pred = final_rf.predict(X_test)
# Convert predictions back to original price scale
rf_pred = np.expm1(rf_pred)
y_test_original = np.expm1(y_test)

rf_r2 = r2_score(y_test_original, rf_pred)
rf_mae = mean_absolute_error(y_test_original, rf_pred)

print("RF R2:", rf_r2)
print("RF MAE:", rf_mae)

RF R2: 0.8904979140900474
RF MAE: 0.40095694360867357


In [75]:
## finally choosing XGBOOST 
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'floor_category','NearbyLocationsCategory']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant_room', 'store_room','study_room']),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value',
        unknown_value=-1), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore'),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [76]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', xgb)
])

In [79]:
import pickle
from sklearn.base import clone

# Create a XGBoost pipeline
final_xgb_v1 = clone(pipeline)

# Apply the best Bayesian optimization parameters
final_xgb_v1.set_params(**bayes_search.best_params_)

# Train on the complete dataset
final_xgb_v1.fit(X, y_transformed)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [80]:
with open('xgboost_pipeline.pkl', 'wb') as file:
    pickle.dump(final_xgb_v1, file)

In [81]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

## Experimenting with XGBOOST to find few more best params

In [83]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.model_selection import KFold

search_spaces_2 = {
    'regressor__n_estimators': Integer(200, 1000),
    'regressor__max_depth': Integer(3, 10),
    'regressor__learning_rate': Real(0.01, 0.2, prior='log-uniform'),

    'regressor__subsample': Real(0.6, 1.0),
    'regressor__colsample_bytree': Real(0.5, 1.0),

    'regressor__min_child_weight': Integer(1, 10),
    'regressor__gamma': Real(0, 5),

    'regressor__reg_alpha': Real(1e-5, 10, prior='log-uniform'),
    'regressor__reg_lambda': Real(1, 30, prior='log-uniform'),

    'regressor__max_leaves': Integer(0, 64),
    'regressor__max_bin': Integer(128, 512),

    'regressor__grow_policy': Categorical(['depthwise', 'lossguide'])
}

In [84]:
xgb2 = XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse',
    tree_method='hist',
    random_state=42,
    n_jobs=1
)

In [85]:
kfold2 = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

bayes_search2 = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_spaces_2,
    n_iter=100,
    cv=kfold2,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
    verbose=2
)

In [86]:
bayes_search2.fit(X_train, y_train)

Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for each of 1 candidates, totalling 10 fits
Fitting 10 folds for eac

,estimator,"Pipeline(step...=None, ...))])"
,search_spaces,"{'regressor__colsample_bytree': Real(low=0.5,...m='normalize'), 'regressor__gamma': Real(low=0, h...m='normalize'), 'regressor__grow_policy': Categorical(c...), prior=None), 'regressor__learning_rate': Real(low=0.01...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,100
,scoring,'r2'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,KFold(n_split... shuffle=True)


In [87]:
print("Best Parameters:", bayes_search2.best_params_)
print("Best CV R2:", bayes_search2.best_score_)

Best Parameters: OrderedDict([('regressor__colsample_bytree', 0.6866392112884995), ('regressor__gamma', 0.0), ('regressor__grow_policy', 'depthwise'), ('regressor__learning_rate', 0.048215543577064236), ('regressor__max_bin', 512), ('regressor__max_depth', 10), ('regressor__max_leaves', 64), ('regressor__min_child_weight', 1), ('regressor__n_estimators', 513), ('regressor__reg_alpha', 0.00044808758137083766), ('regressor__reg_lambda', 1.0), ('regressor__subsample', 1.0)])
Best CV R2: 0.9094287091515726


In [88]:
final_xgb_2 = clone(pipeline)

final_xgb_2.set_params(**bayes_search2.best_params_)

final_xgb_2.fit(X_train, y_train)

xgb_pred = final_xgb_2.predict(X_test)

# Original price scale
xgb_pred = np.expm1(xgb_pred)
y_test_original = np.expm1(y_test)

xgb_r2 = r2_score(y_test_original, xgb_pred)
xgb_mae = mean_absolute_error(y_test_original, xgb_pred)

print("XGBoost Test R2:", xgb_r2)
print("XGBoost Test MAE:", xgb_mae)

XGBoost Test R2: 0.9145817966690949
XGBoost Test MAE: 0.3656677516792729


In [89]:
import pickle
from sklearn.base import clone

# Create final model
best_xgb_model = clone(pipeline)

# Apply best parameters
best_xgb_model.set_params(**bayes_search2.best_params_)

# Train on complete dataset
best_xgb_model.fit(X, y_transformed)

# Export model
with open('best_xgboost_pipeline.pkl', 'wb') as file:
    pickle.dump(best_xgb_model, file)
